In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from arch import arch_model
import roughpy as rp
from tqdm.auto import tqdm

c:\Users\kyler\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASET_DIR = Path("../data/datasets")
FEATURE_DIR = Path("../data/features")

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
DATASET_NAME = "monthly_ff5"

INPUT_DIR = DATASET_DIR / DATASET_NAME
OUTPUT_DIR = FEATURE_DIR / DATASET_NAME

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
X_train = np.load(INPUT_DIR / "X_train.npy")
X_validation = np.load(INPUT_DIR / "X_validation.npy")
X_test = np.load(INPUT_DIR / "X_test.npy")

y_train = np.load(INPUT_DIR / "y_train.npy")
y_validation = np.load(INPUT_DIR / "y_validation.npy")
y_test = np.load(INPUT_DIR / "y_test.npy")

meta_train = pd.read_csv(
    INPUT_DIR / "meta_train.csv",
    parse_dates=["Date"]
)

meta_validation = pd.read_csv(
    INPUT_DIR / "meta_validation.csv",
    parse_dates=["Date"]
)

meta_test = pd.read_csv(
    INPUT_DIR / "meta_test.csv",
    parse_dates=["Date"]
)

In [5]:
print("Training:", X_train.shape, y_train.shape, meta_train.shape)
print("Validation:", X_validation.shape, y_validation.shape, meta_validation.shape)
print("Testing:", X_test.shape, y_test.shape, meta_test.shape)

print("\nExample window:")
print(X_train[0])

print("\nExample metadata:")
display(meta_train.head())

Training: (12950, 12, 7) (12950,) (12950, 4)
Validation: (2775, 12, 7) (2775,) (2775, 4)
Testing: (2800, 12, 7) (2800,) (2800, 4)

Example window:
[[ 1.1287 -0.39   -0.48   -0.81    0.64   -1.15    0.27  ]
 [ 4.2396  5.08   -0.8     1.7     0.4    -0.38    0.25  ]
 [-1.7343 -1.57   -0.43    0.     -0.78    0.15    0.27  ]
 [ 0.3778  2.54   -1.34   -0.04    2.79   -2.25    0.29  ]
 [-3.3319 -0.86   -0.85    1.73   -0.43    2.27    0.27  ]
 [-2.3435  1.83   -1.89   -0.21    0.12   -0.25    0.29  ]
 [ 3.8925  2.27    0.1     1.63    0.21    1.48    0.3   ]
 [ 2.8826  1.55    0.33    2.81    0.11    0.81    0.26  ]
 [ 0.4753  1.41    1.41    3.29   -2.03    2.98    0.31  ]
 [-1.5767  0.11   -1.48   -0.54   -1.32   -1.13    0.29  ]
 [-1.239   1.41   -0.62    1.81   -0.15    0.13    0.26  ]
 [ 1.9444  1.27    0.13    0.68   -0.33    0.1     0.3   ]]

Example metadata:


,Date,RealizationDate,Portfolio,RealizedNextReturn
0,1964-07-01,1964-08-01,SMALL LoBM,1.6973
1,1964-07-01,1964-08-01,ME1 BM2,-2.5518
2,1964-07-01,1964-08-01,ME1 BM3,-1.7484
3,1964-07-01,1964-08-01,ME1 BM4,-0.8049
4,1964-07-01,1964-08-01,SMALL HiBM,-0.4831


In [6]:
print("RoughPy version:", rp.__version__)
print(dir(rp))

RoughPy version: 0.3.0
['BrownianStream', 'CategoricalChannel', 'ChannelType', 'Clopen', 'Context', 'DPReal', 'DateTimeInterval', 'DenseVector', 'Dyadic', 'DyadicInterval', 'ExternalDataStream', 'FreeTensor', 'FreeTensorIteratorItem', 'FunctionStream', 'HPReal', 'IncrementChannel', 'Interval', 'IntervalType', 'LIBS_DIR', 'Lie', 'LieBasis', 'LieChannel', 'LieIncrementStream', 'LieIteratorItem', 'LieKey', 'LieKeyIterator', 'Monomial', 'Partition', 'PiecewiseAbelianStream', 'PolynomialScalar', 'Rational', 'RationalPoly', 'RealInterval', 'SCALAR_MAPPING', 'SPReal', 'Scalar', 'ScalarMeta', 'ScalarTypeBase', 'ShuffleTensor', 'ShuffleTensorIteratorItem', 'SparseVector', 'Stream', 'StreamInterface', 'StreamSchema', 'TensorBasis', 'TensorKey', 'TensorKeyIterator', 'TensorValuedStream', 'TickStream', 'TickStreamConstructionHelper', 'ValueChannel', 'VectorType', '_Path', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__versio

In [7]:
def add_time_channel(sequence):
    """
    Add normalized time as the first channel.

    Parameters
    ----------
    sequence : ndarray
        Shape (window_length, n_features)

    Returns
    -------
    ndarray
        Shape (window_length, n_features + 1)
    """

    sequence = np.asarray(sequence, dtype=np.float64)

    time = np.linspace(
        0.0,
        1.0,
        sequence.shape[0],
        dtype=np.float64
    ).reshape(-1, 1)

    return np.concatenate([time, sequence], axis=1)

In [8]:
example_path = add_time_channel(X_train[0])

print("Original shape:", X_train[0].shape)
print("Path shape:", example_path.shape)
print(example_path[:3])

Original shape: (12, 7)
Path shape: (12, 8)
[[ 0.          1.12870002 -0.38999999 -0.47999999 -0.81        0.63999999
  -1.14999998  0.27000001]
 [ 0.09090909  4.23960018  5.07999992 -0.80000001  1.70000005  0.40000001
  -0.38        0.25      ]
 [ 0.18181818 -1.73430002 -1.57000005 -0.43000001  0.         -0.77999997
   0.15000001  0.27000001]]


In [9]:
SIGNATURE_DEPTH = 2

In [10]:
def roughpy_logsignature(sequence, depth=2):
    """
    Compute a flattened RoughPy log-signature for one sequence.
    """

    path_values = add_time_channel(sequence)

    width = path_values.shape[1]

    stream = rp.LieIncrementStream.from_increments(
        path_values,
        width=width,
        depth=depth,
        dtype=rp.DPReal
    )

    log_signature = stream.log_signature()

    return np.asarray(log_signature, dtype=np.float64).reshape(-1)

In [11]:
test_logsig = roughpy_logsignature(
    X_train[0],
    depth=SIGNATURE_DEPTH
)

print("Log-signature shape:", test_logsig.shape)
print(test_logsig[:10])

Log-signature shape: (36,)
[  6.           4.71550041  14.64999972  -5.9200001   12.0499999
  -0.77000008   2.76000006   3.36000001  -5.77100096 -16.44363602]


C:\Users\kyler\AppData\Local\Temp\ipykernel_21136\1074427596.py:10: DeprecationWarning: using the "width" keyword argument is deprecated, instead you should use a context and the "ctx" keyword argument to specify the algebra configuration
  stream = rp.LieIncrementStream.from_increments(


In [12]:
print(dir(rp))

['BrownianStream', 'CategoricalChannel', 'ChannelType', 'Clopen', 'Context', 'DPReal', 'DateTimeInterval', 'DenseVector', 'Dyadic', 'DyadicInterval', 'ExternalDataStream', 'FreeTensor', 'FreeTensorIteratorItem', 'FunctionStream', 'HPReal', 'IncrementChannel', 'Interval', 'IntervalType', 'LIBS_DIR', 'Lie', 'LieBasis', 'LieChannel', 'LieIncrementStream', 'LieIteratorItem', 'LieKey', 'LieKeyIterator', 'Monomial', 'Partition', 'PiecewiseAbelianStream', 'PolynomialScalar', 'Rational', 'RationalPoly', 'RealInterval', 'SCALAR_MAPPING', 'SPReal', 'Scalar', 'ScalarMeta', 'ScalarTypeBase', 'ShuffleTensor', 'ShuffleTensorIteratorItem', 'SparseVector', 'Stream', 'StreamInterface', 'StreamSchema', 'TensorBasis', 'TensorKey', 'TensorKeyIterator', 'TensorValuedStream', 'TickStream', 'TickStreamConstructionHelper', 'ValueChannel', 'VectorType', '_Path', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_add_dynload_loc

In [13]:
def create_logsignature_features(X, depth=2, description="Log-signatures"):
    """
    Convert a collection of rolling windows into log-signature vectors.
    """

    features = []

    for sequence in tqdm(X, desc=description):
        feature_vector = roughpy_logsignature(
            sequence,
            depth=depth
        )

        features.append(feature_vector)

    return np.vstack(features)

In [14]:
X_signature_test_batch = create_logsignature_features(
    X_train[:10],
    depth=SIGNATURE_DEPTH,
    description="Testing RoughPy"
)

print(X_signature_test_batch.shape)

Testing RoughPy:   0%|          | 0/10 [00:00<?, ?it/s]C:\Users\kyler\AppData\Local\Temp\ipykernel_21136\1074427596.py:10: DeprecationWarning: using the "width" keyword argument is deprecated, instead you should use a context and the "ctx" keyword argument to specify the algebra configuration
  stream = rp.LieIncrementStream.from_increments(
Testing RoughPy: 100%|██████████| 10/10 [00:00<00:00, 3402.26it/s]

(10, 36)


In [15]:
SIGNATURE_DEPTH = 2

In [16]:
def roughpy_logsignature(sequence, depth=2):
    """
    Compute a flattened RoughPy log-signature for one rolling window.

    Parameters
    ----------
    sequence : np.ndarray
        Shape: (window_length, n_features)

    depth : int
        Log-signature truncation depth.

    Returns
    -------
    np.ndarray
        Flattened log-signature feature vector.
    """

    path = add_time_channel(sequence)

    # Convert observations into path increments
    increments = np.diff(path, axis=0)

    stream = rp.LieIncrementStream.from_increments(
        increments,
        depth=depth
    )

    interval = rp.RealInterval(
        0.0,
        float(len(increments))
    )

    log_signature = stream.log_signature(interval)

    return np.asarray(log_signature).reshape(-1)

In [17]:
test_logsig = roughpy_logsignature(
    X_train[0],
    depth=SIGNATURE_DEPTH
)

print("Log-signature shape:", test_logsig.shape)
print("First values:", test_logsig[:10])

Log-signature shape: (36,)
First values: [ 1.          0.81569993  1.65999997  0.60999998  1.49000001 -0.97
  1.24999998  0.03        1.24755449 -0.85181816]


C:\Users\kyler\AppData\Local\Temp\ipykernel_21136\4154849035.py:24: DeprecationWarning: using the "depth" keyword argument is deprecated, instead you should use a context and the "ctx" keyword argument to specify the algebra configuration
  stream = rp.LieIncrementStream.from_increments(


In [18]:
for i in range(5):
    feature = roughpy_logsignature(
        X_train[i],
        depth=SIGNATURE_DEPTH
    )

    print(i, feature.shape)

0 (36,)
1 (36,)
2 (36,)
3 (36,)
4 (36,)


C:\Users\kyler\AppData\Local\Temp\ipykernel_21136\4154849035.py:24: DeprecationWarning: using the "depth" keyword argument is deprecated, instead you should use a context and the "ctx" keyword argument to specify the algebra configuration
  stream = rp.LieIncrementStream.from_increments(


In [19]:
X_signature_train = create_logsignature_features(
    X_train,
    depth=SIGNATURE_DEPTH,
    description="Monthly FF5 train signatures"
)

X_signature_validation = create_logsignature_features(
    X_validation,
    depth=SIGNATURE_DEPTH,
    description="Monthly FF5 validation signatures"
)

X_signature_test = create_logsignature_features(
    X_test,
    depth=SIGNATURE_DEPTH,
    description="Monthly FF5 test signatures"
)

Monthly FF5 train signatures:   0%|          | 0/12950 [00:00<?, ?it/s]C:\Users\kyler\AppData\Local\Temp\ipykernel_21136\4154849035.py:24: DeprecationWarning: using the "depth" keyword argument is deprecated, instead you should use a context and the "ctx" keyword argument to specify the algebra configuration
  stream = rp.LieIncrementStream.from_increments(
Monthly FF5 test signatures: 100%|██████████| 2800/2800 [00:00<00:00, 8281.92it/s]


In [20]:
print("Signature train:", X_signature_train.shape)
print("Signature validation:", X_signature_validation.shape)
print("Signature test:", X_signature_test.shape)

print("Missing train values:", np.isnan(X_signature_train).sum())
print("Infinite train values:", np.isinf(X_signature_train).sum())

Signature train: (12950, 36)
Signature validation: (2775, 36)
Signature test: (2800, 36)
Missing train values: 0
Infinite train values: 0


In [21]:
np.save(
    OUTPUT_DIR / "X_signature_train.npy",
    X_signature_train
)

np.save(
    OUTPUT_DIR / "X_signature_validation.npy",
    X_signature_validation
)

np.save(
    OUTPUT_DIR / "X_signature_test.npy",
    X_signature_test
)

print("Saved RoughPy features to:", OUTPUT_DIR.resolve())

Saved RoughPy features to: C:\Users\kyler\Documents\VS_Code\Finance Code\ML using FAMA and FRENCH\data\features\monthly_ff5


## GARCH Section

In [22]:
GARCH_FEATURE_NAMES = [
    "garch_mu",
    "garch_omega",
    "garch_alpha",
    "garch_beta",
    "garch_persistence",
    "conditional_volatility_last",
    "conditional_volatility_mean",
    "conditional_volatility_std",
    "conditional_volatility_min",
    "conditional_volatility_max",
    "standardized_residual_last",
    "absolute_residual_last"
]

In [23]:
def extract_garch_features(sequence):
    """
    Fit GARCH(1,1) to the portfolio-return channel of one window.

    Returns a fixed-length vector of volatility features.
    """

    sequence = np.asarray(sequence, dtype=np.float64)

    # First channel is portfolio return
    returns = sequence[:, 0]

    # ARCH generally behaves better when returns are expressed in percent
    returns_pct = returns * 100.0

    try:
        model = arch_model(
            returns_pct,
            mean="Constant",
            vol="GARCH",
            p=1,
            q=1,
            dist="normal",
            rescale=False
        )

        result = model.fit(
            disp="off",
            show_warning=False
        )

        parameters = result.params
        conditional_volatility = np.asarray(
            result.conditional_volatility,
            dtype=np.float64
        )

        residuals = np.asarray(
            result.resid,
            dtype=np.float64
        )

        standardized_residuals = (
            residuals / conditional_volatility
        )

        mu = parameters.get("mu", np.nan)
        omega = parameters.get("omega", np.nan)
        alpha = parameters.get("alpha[1]", np.nan)
        beta = parameters.get("beta[1]", np.nan)

        features = np.array([
            mu,
            omega,
            alpha,
            beta,
            alpha + beta,
            conditional_volatility[-1],
            conditional_volatility.mean(),
            conditional_volatility.std(),
            conditional_volatility.min(),
            conditional_volatility.max(),
            standardized_residuals[-1],
            abs(residuals[-1])
        ], dtype=np.float64)

    except Exception:
        features = np.full(
            len(GARCH_FEATURE_NAMES),
            np.nan,
            dtype=np.float64
        )

    return features

In [24]:
test_garch = extract_garch_features(X_train[0])

print("GARCH feature shape:", test_garch.shape)

display(
    pd.Series(
        test_garch,
        index=GARCH_FEATURE_NAMES,
        name="value"
    )
)

GARCH feature shape: (12,)


garch_mu                          39.295688
garch_omega                    17156.974964
garch_alpha                        0.000000
garch_beta                         0.693502
garch_persistence                  0.693502
conditional_volatility_last      236.692436
conditional_volatility_mean      238.041636
conditional_volatility_std         1.595246
conditional_volatility_min       236.692436
conditional_volatility_max       241.956172
standardized_residual_last         0.655468
absolute_residual_last           155.144307
Name: value, dtype: float64

In [25]:
MONTHLY_WINDOW = 60
DAILY_WINDOW = 252

In [26]:
def create_garch_features(X, description="GARCH features"):
    """
    Generate one GARCH feature vector per rolling window.
    """

    features = []

    for sequence in tqdm(X, desc=description):
        feature_vector = extract_garch_features(sequence)
        features.append(feature_vector)

    return np.vstack(features)

In [27]:
X_garch_test_batch = create_garch_features(
    X_train[:20],
    description="Testing GARCH"
)

print(X_garch_test_batch.shape)

display(
    pd.DataFrame(
        X_garch_test_batch,
        columns=GARCH_FEATURE_NAMES
    ).head()
)

Testing GARCH: 100%|██████████| 20/20 [00:00<00:00, 104.66it/s]

(20, 12)


,garch_mu,garch_omega,garch_alpha,garch_beta,garch_persistence,conditional_volatility_last,conditional_volatility_mean,conditional_volatility_std,conditional_volatility_min,conditional_volatility_max,standardized_residual_last,absolute_residual_last
0,39.295688,17156.974964,0.000000e+00,0.693502,0.693502,236.692436,238.041636,1.595246,236.692436,241.956172,0.655468,155.144307
1,42.411417,18076.184340,0.000000e+00,0.531821,0.531821,196.488973,195.731853,1.247753,192.197896,196.488973,0.001876,0.368583
2,76.886391,12988.974963,1.780501e-01,0.326281,0.504331,145.272985,162.942110,17.738659,144.667920,208.633259,0.302559,43.953610
3,57.298592,16866.332668,3.816392e-17,0.520882,0.520882,187.622517,187.207977,0.695796,185.224252,187.622517,0.122328,22.951409
4,125.990252,25731.088170,5.089786e-01,0.000000,0.508979,160.561726,228.984973,44.033051,160.561726,308.300271,-0.078725,12.640254


In [28]:
failed_rows = np.isnan(X_garch_test_batch).any(axis=1)

print("Successful fits:", (~failed_rows).sum())
print("Failed fits:", failed_rows.sum())
print("Failure rate:", failed_rows.mean())

Successful fits: 20
Failed fits: 0
Failure rate: 0.0


In [29]:
X_garch_train = create_garch_features(
    X_train,
    description="Monthly FF5 train GARCH"
)

X_garch_validation = create_garch_features(
    X_validation,
    description="Monthly FF5 validation GARCH"
)

X_garch_test = create_garch_features(
    X_test,
    description="Monthly FF5 test GARCH"
)

Monthly FF5 test GARCH: 100%|██████████| 2800/2800 [00:24<00:00, 112.86it/s]


In [30]:
print("GARCH train:", X_garch_train.shape)
print("GARCH validation:", X_garch_validation.shape)
print("GARCH test:", X_garch_test.shape)

GARCH train: (12950, 12)
GARCH validation: (2775, 12)
GARCH test: (2800, 12)


In [31]:
def summarize_garch_failures(X_garch, split_name):
    failed_rows = np.isnan(X_garch).any(axis=1)

    print(split_name)
    print("Total samples:", len(X_garch))
    print("Failed fits:", failed_rows.sum())
    print("Failure rate:", failed_rows.mean())
    print()

    return failed_rows

In [32]:
failed_train = summarize_garch_failures(
    X_garch_train,
    "Train"
)

failed_validation = summarize_garch_failures(
    X_garch_validation,
    "Validation"
)

failed_test = summarize_garch_failures(
    X_garch_test,
    "Test"
)

Train
Total samples: 12950
Failed fits: 0
Failure rate: 0.0

Validation
Total samples: 2775
Failed fits: 0
Failure rate: 0.0

Test
Total samples: 2800
Failed fits: 0
Failure rate: 0.0



In [33]:
def fit_median_imputer(X_train):
    """
    Compute training-set medians for each feature.
    """
    medians = np.nanmedian(X_train, axis=0)

    # If an entire feature column is NaN, replace its median with zero
    medians = np.where(np.isnan(medians), 0.0, medians)

    return medians

In [34]:
def apply_median_imputer(X, medians):
    """
    Replace NaN and infinite values using training-set medians.
    """
    X = np.asarray(X, dtype=np.float64).copy()

    X[~np.isfinite(X)] = np.nan

    missing_rows, missing_cols = np.where(np.isnan(X))

    X[missing_rows, missing_cols] = medians[missing_cols]

    return X

In [35]:
garch_medians = fit_median_imputer(X_garch_train)

X_garch_train_clean = apply_median_imputer(
    X_garch_train,
    garch_medians
)

X_garch_validation_clean = apply_median_imputer(
    X_garch_validation,
    garch_medians
)

X_garch_test_clean = apply_median_imputer(
    X_garch_test,
    garch_medians
)

In [36]:
print("Train missing:", np.isnan(X_garch_train_clean).sum())
print("Validation missing:", np.isnan(X_garch_validation_clean).sum())
print("Test missing:", np.isnan(X_garch_test_clean).sum())

Train missing: 0
Validation missing: 0
Test missing: 0


In [37]:
from sklearn.preprocessing import StandardScaler

In [38]:
garch_scaler = StandardScaler()

X_garch_train_scaled = garch_scaler.fit_transform(
    X_garch_train_clean
)

X_garch_validation_scaled = garch_scaler.transform(
    X_garch_validation_clean
)

X_garch_test_scaled = garch_scaler.transform(
    X_garch_test_clean
)

In [39]:
print("Training means:")
print(np.round(X_garch_train_scaled.mean(axis=0), 4))

print("\nTraining standard deviations:")
print(np.round(X_garch_train_scaled.std(axis=0), 4))

Training means:
[-0.  0.  0.  0.  0.  0. -0.  0.  0.  0. -0.  0.]

Training standard deviations:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [40]:
def clean_feature_array(X):
    X = np.asarray(X, dtype=np.float64).copy()
    X[~np.isfinite(X)] = np.nan
    return X

In [41]:
X_signature_train_clean = clean_feature_array(
    X_signature_train
)

X_signature_validation_clean = clean_feature_array(
    X_signature_validation
)

X_signature_test_clean = clean_feature_array(
    X_signature_test
)

In [42]:
signature_medians = fit_median_imputer(
    X_signature_train_clean
)

X_signature_train_clean = apply_median_imputer(
    X_signature_train_clean,
    signature_medians
)

X_signature_validation_clean = apply_median_imputer(
    X_signature_validation_clean,
    signature_medians
)

X_signature_test_clean = apply_median_imputer(
    X_signature_test_clean,
    signature_medians
)

In [43]:
signature_scaler = StandardScaler()

X_signature_train_scaled = signature_scaler.fit_transform(
    X_signature_train_clean
)

X_signature_validation_scaled = signature_scaler.transform(
    X_signature_validation_clean
)

X_signature_test_scaled = signature_scaler.transform(
    X_signature_test_clean
)

In [44]:
X_combined_train = np.concatenate(
    [
        X_signature_train_scaled,
        X_garch_train_scaled
    ],
    axis=1
)

X_combined_validation = np.concatenate(
    [
        X_signature_validation_scaled,
        X_garch_validation_scaled
    ],
    axis=1
)

X_combined_test = np.concatenate(
    [
        X_signature_test_scaled,
        X_garch_test_scaled
    ],
    axis=1
)

In [45]:
print("Signature:", X_signature_train_scaled.shape)
print("GARCH:", X_garch_train_scaled.shape)
print("Combined:", X_combined_train.shape)

assert X_combined_train.shape[0] == len(y_train)
assert X_combined_validation.shape[0] == len(y_validation)
assert X_combined_test.shape[0] == len(y_test)

Signature: (12950, 36)
GARCH: (12950, 12)
Combined: (12950, 48)


In [46]:
np.save(
    OUTPUT_DIR / "X_signature_train.npy",
    X_signature_train_scaled
)

np.save(
    OUTPUT_DIR / "X_signature_validation.npy",
    X_signature_validation_scaled
)

np.save(
    OUTPUT_DIR / "X_signature_test.npy",
    X_signature_test_scaled
)

np.save(
    OUTPUT_DIR / "X_garch_train.npy",
    X_garch_train_scaled
)

np.save(
    OUTPUT_DIR / "X_garch_validation.npy",
    X_garch_validation_scaled
)

np.save(
    OUTPUT_DIR / "X_garch_test.npy",
    X_garch_test_scaled
)

np.save(
    OUTPUT_DIR / "X_combined_train.npy",
    X_combined_train
)

np.save(
    OUTPUT_DIR / "X_combined_validation.npy",
    X_combined_validation
)

np.save(
    OUTPUT_DIR / "X_combined_test.npy",
    X_combined_test
)

In [47]:
np.save(OUTPUT_DIR / "y_train.npy", y_train)
np.save(OUTPUT_DIR / "y_validation.npy", y_validation)
np.save(OUTPUT_DIR / "y_test.npy", y_test)

meta_train.to_csv(
    OUTPUT_DIR / "meta_train.csv",
    index=False
)

meta_validation.to_csv(
    OUTPUT_DIR / "meta_validation.csv",
    index=False
)

meta_test.to_csv(
    OUTPUT_DIR / "meta_test.csv",
    index=False
)

In [48]:
import joblib

In [49]:
preprocessing_objects = {
    "signature_depth": SIGNATURE_DEPTH,
    "signature_medians": signature_medians,
    "garch_medians": garch_medians,
    "signature_scaler": signature_scaler,
    "garch_scaler": garch_scaler,
    "garch_feature_names": GARCH_FEATURE_NAMES
}

joblib.dump(
    preprocessing_objects,
    OUTPUT_DIR / "preprocessing.joblib"
)

['..\\data\\features\\monthly_ff5\\preprocessing.joblib']

In [50]:
saved_arrays = {
    "signature_train": OUTPUT_DIR / "X_signature_train.npy",
    "garch_train": OUTPUT_DIR / "X_garch_train.npy",
    "combined_train": OUTPUT_DIR / "X_combined_train.npy",
    "y_train": OUTPUT_DIR / "y_train.npy"
}

for name, path in saved_arrays.items():
    array = np.load(path)

    print(
        f"{name:20} "
        f"shape={array.shape}, "
        f"missing={np.isnan(array).sum()}"
    )

signature_train      shape=(12950, 36), missing=0
garch_train          shape=(12950, 12), missing=0
combined_train       shape=(12950, 48), missing=0
y_train              shape=(12950,), missing=0
